

### Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---

**🔐 Як безпечно зберігати і підвантажувати API-ключі**

API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.



In [76]:
!pip -q install langchain langchain_openai langchain-classic

In [77]:
import json
import os
from langchain_openai import OpenAI
from langchain_core.prompts import PromptTemplate

In [78]:
with open('creds.json') as file:
  creds = json.load(file)

os.environ["OPENAI_API_KEY"] = creds["OPENAI_API_KEY"]
overal_temperature = 0.5 # I want something in the middle not too variable temperature but not to concervvative

In [79]:
llm = OpenAI(temperature=overal_temperature)

In [80]:
request = "What is quantum computing? Be concise in answer, and keep response less than 200 chars."

for chunk in llm.stream(
    request
):
  print(chunk, end="", flush=True)



Quantum computing is a form of computing that utilizes the principles of quantum mechanics to process and store information, potentially allowing for much faster and more powerful calculations than traditional computers.

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [81]:
prompt = PromptTemplate(
    input_variables=["topic"],
    template="Explain next topic: {topic}? Be concise in answer, and keep response less than 200 chars.",
)

In [82]:
prompt_topic="Bayesian methods in machine learning"
print(prompt.format(topic=prompt_topic))

print(llm.invoke(prompt.format(topic=prompt_topic)))

Explain next topic: Bayesian methods in machine learning? Be concise in answer, and keep response less than 200 chars.


Bayesian methods in machine learning use probability theory to make predictions and decisions based on prior knowledge and new data. They allow for uncertainty and can update beliefs as new information is received.


In [83]:
prompt_topic="Transformers in machine learning"
print(prompt.format(topic=prompt_topic))

print(llm.invoke(prompt.format(topic=prompt_topic)))

Explain next topic: Transformers in machine learning? Be concise in answer, and keep response less than 200 chars.


Transformers in machine learning are a type of neural network architecture that uses self-attention mechanisms to process sequential data, such as text. They have achieved state-of-the-art results in natural language processing tasks.


In [84]:
prompt_topic="Explainable AI"
print(prompt.format(topic=prompt_topic))

print(llm.invoke(prompt.format(topic=prompt_topic)))

Explain next topic: Explainable AI? Be concise in answer, and keep response less than 200 chars.


Explainable AI (XAI) is the concept of developing artificial intelligence systems that can provide understandable explanations for their decisions and actions, allowing humans to understand and trust the technology. This is important for increasing transparency, accountability, and ethical considerations in AI.




### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [85]:
!pip install -q langchain_community duckduckgo_search
!pip install -q ddgs
!pip install -q mypy-extensions

In [86]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

'The first president, George Washington, won a unanimous vote of the Electoral College.[4] The incumbent president is Donald Trump, who assumed office on January 20, 2025.[5][6]... Malia Obama. First Daughter of the United States."Obama\'s Friends Form Strategy to Stay Close". The New York Times. ISSN 0362-4331. Barack Obama, the 44th President of the United States, broke barriers as the first African-American president and implemented significant healthcare reforms during his tenure. Barack Obama was the 44th president of the United States (2009-2017) and the first African American to be elected to ... Barack Obama Barack Obama was the 44th president of the United States...'

In [87]:
from langchain.agents import create_agent
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

In [88]:
tools = [search]

llm = ChatOpenAI(model="gpt-4o", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant tasked with finding information about scientific publications. Respond in Ukrainian."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(
    llm,
    tools,
    prompt
)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

agent_request = (
    "Знайди 5 останніх наукових публікацій на тему 'штучний інтелект'. "
    "Для кожної публікації, будь ласка, надайте: назву, авторів і короткий опис."
)

print(f"\nExecuting agent with request: {agent_request}\n")
result = agent_executor.invoke({"input": agent_request})

print("\nAgent's final response:")
print(result["output"])


Executing agent with request: Знайди 5 останніх наукових публікацій на тему 'штучний інтелект'. Для кожної публікації, будь ласка, надайте: назву, авторів і короткий опис.



> Entering new AgentExecutor chain...

Invoking: `duckduckgo_search` with `{'query': 'latest scientific publications on artificial intelligence 2023'}`


Subjects: Artificial Intelligence (cs.AI); Computer Vision and Pattern Recognition (cs.CV); Information Theory (cs.IT) The annual report tracks, collates, distills, and visualizes data relating to artificial intelligence, enabling decision-makers to take meaningful action to advance AI responsibly and ethically with humans in mind. The AI Index 2023 Annual Report by Stanford University is licensed under Attribution-NoDerivatives 4.0 International. Read the latest articles of Artificial Intelligence at ScienceDirect.com, Elsevier’s leading platform of peer-reviewed scholarly literature Aug 2, 2023 · Here we examine breakthroughs over the past decade that include 



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [89]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Combine search tool with the new python_repl tool
tools_business_agent = [search]

# Agent Prompt for the Business Analytics Agent
business_agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "Ви — досвідчений бізнес-аналітик, який допомагає прогнозувати продажі та аналізувати ринок. "
               "Ви можете використовувати інтернет для пошуку актуальних даних (інфляція, погодні умови, попит). "
               "Відповідайте українською. "
               "Якщо ви не можете надати точний прогноз, поясніть чому."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create the Business Analytics Agent
business_agent = create_tool_calling_agent(
    llm,
    tools_business_agent,
    business_agent_prompt
)

# Create the Agent Executor
business_agent_executor = AgentExecutor(
    agent=business_agent,
    tools=tools_business_agent,
    verbose=True,
    handle_parsing_errors=True # Added to catch parsing issues in agent's thoughts
)

# Define the business request from the user
business_request = (
    "Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, "
    "в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати "
    "апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі "
    "виходячи з економічної ситуації."
)

print(f"\nExecuting business analytics agent with request: {business_request}\n")
# For chat_history, we can start with an empty list for the first turn
business_result = business_agent_executor.invoke({"input": business_request, "chat_history": []})

print("\nAgent's final response:")
print(business_result["output"])


Executing business analytics agent with request: Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.



> Entering new AgentExecutor chain...

Invoking: `duckduckgo_search` with `{'query': 'weather conditions Brazil 2026 forecast'}`


Brazil ☔ 7 days detailed forecast. 10, 20 and 30 days long-term weather and climate forecast. Current weather and long-term 45 days forecast ☃ Weather phenomena recorded in previous years in Brazil ☀ Detailed weather forecast for the next 10 days ☔ Long term weather forecast for Brazil for 30 days Fig. 4: Nino34 SSTA forecasts utilizing all models reveal El Nino ahead in 2026. February 2026: The late meteorological summer 2025-26 climate pattern across South America features anomalous dry and hot conditions for Southeast Brazil, and signific